# Code mẫu — Activation và loss

Activation quyết định tín hiệu, và cả gradient, đi qua mạng ra sao. Loss quyết định thế nào là một dự đoán sai, và sai bao nhiêu. Các ví dụ dưới đây dùng những con số nhỏ, kiểm được bằng tay, để xem độ dốc, đơn vị, logit và cách gộp loss.

## Ví dụ 1: Cùng đầu vào, mỗi activation truyền gradient một kiểu

Lấy năm giá trị trước activation, từ −4 tới 4. Ta cộng tất cả đầu ra lại làm hàm mục tiêu. Nhờ vậy, gradient tại mỗi điểm chính là độ dốc của activation ở điểm đó.

In [ ]:
import torch
from torch.nn import functional as F

torch.set_num_threads(1)
values = torch.tensor([-4., -1., 0., 1., 4.], dtype=torch.float64)
for name, activation in [("ReLU", F.relu),
                         ("LeakyReLU", lambda x: F.leaky_relu(x, .1)),
                         ("sigmoid", torch.sigmoid)]:
    z = values.clone().requires_grad_()
    output = activation(z)
    output.sum().backward()
    print(name, "output:", [round(v, 5) for v in output.tolist()])
    print(name, "gradient:", [round(v, 5) for v in z.grad.tolist()])

ReLU output: [0.0, 0.0, 0.0, 1.0, 4.0]
ReLU gradient: [0.0, 0.0, 0.0, 1.0, 1.0]
LeakyReLU output: [-0.4, -0.1, 0.0, 1.0, 4.0]
LeakyReLU gradient: [0.1, 0.1, 0.1, 1.0, 1.0]
sigmoid output: [0.01799, 0.26894, 0.5, 0.73106, 0.98201]
sigmoid gradient: [0.01766, 0.19661, 0.25, 0.19661, 0.01766]


ReLU cho gradient bằng 0 ở mọi giá trị âm. Một neuron mà lúc nào cũng nhận giá trị âm sẽ không bao giờ được cập nhật. LeakyReLU giữ lại độ dốc nhỏ 0.1 ở phía âm, nên gradient vẫn đi qua được.

Sigmoid có độ dốc lớn nhất là 0.25, ngay tại 0. Ra tới ±4 thì độ dốc chỉ còn khoảng 0.018. Qua nhiều lớp sigmoid nối tiếp, các con số nhỏ này nhân với nhau, và gradient tới được các lớp đầu sẽ rất nhỏ. Riêng tại điểm 0, ReLU bị gãy và không có đạo hàm thật. PyTorch quy ước lấy giá trị 0 ở đó.

## Ví dụ 2: MSE phạt nặng lỗi lớn, và có đơn vị bình phương

Ba dự đoán thời gian có phần dư là −1, 2 và 10 phút. Ta muốn biết mẫu nào đẩy loss lên nhiều nhất. Sau đó đổi cùng dữ liệu ấy sang giây để xem loss thay đổi ra sao.

In [ ]:
import torch
from torch.nn import functional as F

torch.set_num_threads(1)
prediction = torch.tensor([4., 7., 15.])
target = torch.tensor([5., 5., 5.])
assert prediction.shape == target.shape
per_item = F.mse_loss(prediction, target, reduction="none")
mse = per_item.mean()
mae = F.l1_loss(prediction, target)
print("phần dư (phút):", (prediction - target).tolist())
print("MSE từng mẫu:", per_item.tolist())
print("MSE / MAE:", round(mse.item(), 4), round(mae.item(), 4))
print("MSE theo giây:", F.mse_loss(60 * prediction, 60 * target).item())
assert torch.allclose(F.mse_loss(60 * prediction, 60 * target), 3600 * mse)

phần dư (phút): [-1.0, 2.0, 10.0]
MSE từng mẫu: [1.0, 4.0, 100.0]
MSE / MAE: 35.0 4.3333
MSE theo giây: 126000.0


MSE bằng $(1+4+100)/3=35$, đơn vị là phút bình phương. MAE bằng $13/3$ phút. Riêng mẫu cuối đã đóng góp 100 trong tổng 105 của MSE. Vì bình phương, một lỗi lớn có thể lấn át mọi lỗi nhỏ.

Đổi sang giây thì mọi phần dư nhân 60, nên MSE nhân $60^2=3600$, thành 126000. Model không hề tệ đi, chỉ có đơn vị thay đổi. Vì vậy đừng so hai giá trị loss tính theo hai đơn vị khác nhau.

## Ví dụ 3: Logit sai rất tự tin vẫn phải cho loss hữu hạn

Ba logit là −1000, 0 và 1000, đi với nhãn 1, 1 và 0. Hai mẫu ngoài cùng đang sai, và sai rất tự tin. Nếu đổi logit sang xác suất trước, sigmoid sẽ làm tròn thành đúng 0 hoặc 1, và log của 0 là âm vô cùng. Ta cần một công thức BCE tính thẳng từ logit để loss vẫn hữu hạn.

In [ ]:
import torch
from torch.nn import functional as F

torch.set_num_threads(1)
z = torch.tensor([-1000., 0., 1000.], dtype=torch.float64, requires_grad=True)
y = torch.tensor([1., 1., 0.], dtype=torch.float64)
hand = z.clamp_min(0) - z * y + torch.log1p(torch.exp(-z.abs()))
individual = F.binary_cross_entropy_with_logits(z, y, reduction="none")
assert torch.allclose(hand, individual) and torch.isfinite(individual).all()
individual.sum().backward()
sum_gradient = z.grad.clone()
print("loss từng mẫu:", [round(v, 6) for v in individual.tolist()])
print("gradient loss sum:", sum_gradient.tolist())

z.grad = None
mean_loss = F.binary_cross_entropy_with_logits(z, y, reduction="mean")
mean_loss.backward()
print("loss mean:", round(mean_loss.item(), 6))
print("gradient loss mean:", [round(v, 6) for v in z.grad.tolist()])
assert torch.allclose(z.grad * len(y), sum_gradient)

loss từng mẫu: [1000.0, 0.693147, 1000.0]
gradient loss sum: [-1.0, -0.5, 1.0]
loss mean: 666.897716
gradient loss mean: [-0.333333, -0.166667, 0.333333]


Hai mẫu sai có loss bằng 1000, lớn nhưng hữu hạn. Gradient của chúng là −1 và 1, nghĩa là model vẫn được chỉ rõ phải sửa theo hướng nào. Mẫu ở giữa có gradient −0.5, đúng bằng $\sigma(0)-1$. Công thức tự viết `hand` cho cùng giá trị loss với thư viện. Phần backward thì đi qua hàm của thư viện.

Hai khối dưới cho thấy tác động của cách gộp. Lấy `mean` thay cho `sum` thì cả loss lẫn gradient đều chia cho 3. Giữ nguyên learning rate mà đổi cách gộp, bước cập nhật sẽ nhỏ đi ba lần.

## Ví dụ 4: Chọn một trong ba lớp, hay hai thuộc tính độc lập

Một mẫu phải thuộc đúng một trong ba lớp. Logit là $(\log3,\log1,\log2)$, nên softmax cho xác suất $(3/6,1/6,2/6)$. Lớp thật là lớp 0, nên loss bằng $-\log(1/2)=\log2$. Ta so cách gọi đúng với một lỗi hay gặp: đưa xác suất đã softmax vào `cross_entropy`.

In [ ]:
import math
import torch
from torch.nn import functional as F

torch.set_num_threads(1)
logits = torch.tensor([[math.log(3), 0., math.log(2)]], dtype=torch.float64)
label = torch.tensor([0], dtype=torch.long)
probabilities = logits.softmax(dim=1)  # Trục 1 là trục lớp, không phải trục batch.
correct = F.cross_entropy(logits, label)
wrong = F.cross_entropy(probabilities, label)
shifted = F.cross_entropy(logits + 1000, label)
print("xác suất ba lớp:", [[round(v, 6) for v in row]
                            for row in probabilities.tolist()])
print("CE logits / CE sau softmax:", round(correct.item(), 6), round(wrong.item(), 6))
print("CE sau cộng 1000:", round(shifted.item(), 6))
print("lớp dự đoán:", logits.argmax(dim=1).tolist())
assert math.isclose(correct.item(), math.log(2))
assert torch.allclose(correct, shifted)

xác suất ba lớp: [[0.5, 0.166667, 0.333333]]
CE logits / CE sau softmax: 0.693147 0.941184
CE sau cộng 1000: 0.693147
lớp dự đoán: [0]


Gọi đúng thì CE bằng 0.693147, tức $\log 2$. `F.cross_entropy` tự làm softmax bên trong. Nếu đưa vào xác suất, nó sẽ coi các xác suất ấy là logit và softmax thêm lần nữa, nên ra 0.941. Code vẫn chạy, không báo lỗi gì. Cộng cùng 1000 vào mọi logit thì khoảng cách giữa chúng giữ nguyên, nên xác suất và loss đều không đổi.

Có những bài mà các nhãn không loại trừ nhau. Một bài lập trình có thể vừa là bài đồ thị vừa là bài quy hoạch động. Khi đó mỗi thuộc tính là một câu hỏi có/không riêng. Logit $\log4$ ứng với xác suất 0.8 cho từng thuộc tính.

In [ ]:
attribute_logits = torch.tensor([[math.log(4), math.log(4)]], dtype=torch.float64)
attribute_target = torch.tensor([[1., 1.]], dtype=torch.float64)
assert attribute_logits.shape == attribute_target.shape
print("sigmoid từng thuộc tính:", attribute_logits.sigmoid().tolist())
print("softmax ép tổng 1:", attribute_logits.softmax(dim=1).tolist())
print("BCE mean:", round(F.binary_cross_entropy_with_logits(
    attribute_logits, attribute_target).item(), 6))

sigmoid từng thuộc tính: [[0.8, 0.8]]
softmax ép tổng 1: [[0.5, 0.5]]
BCE mean: 0.223144


Sigmoid cho mỗi thuộc tính xác suất 0.8, nên cả hai cùng đúng là chuyện bình thường. Softmax thì ép tổng bằng 1, biến chúng thành 0.5 và 0.5, như thể hai thuộc tính phải giành nhau. Với nhãn nhiều thuộc tính, ta dùng sigmoid cho từng cột cùng với BCE, và nhãn có cùng shape với logit.